# Kubernetes with kubeadm {background-color="black" background-image="https://images.unsplash.com/photo-1667372459567-3853510dd5ce?q=80&w=2532&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D" background-size="cover" background-opacity="0.5"}


# Why Kubernetes? {background-color="#326CE5"}


## From Docker Swarm to Kubernetes

::::{.columns}

::: {.fragment .column width="30%"}

**Docker Swarm**

- A single `docker stack deploy` command deploys a whole app
- Manages a cluster with Manager and Workers
- Great for scaling a single multi-container app on familiar Docker syntax

::: {.callout-note  .fragment}
But what happens when you have more applications, or more complex requirements?
:::

:::

::: {.fragment .column width="70%"}

**Kubernetes**

| Feature | Swarm | Kubernetes |
|---|---|---|
| Workload types | Service | Deployment, StatefulSet, DaemonSet, Job… |
| Scheduling | Basic | Resource requests, affinity, taints |
| Storage | Named volumes | PVC, CSI |
| Networking | Overlay | CNI plugin ecosystem |
| Self-healing | ✅ | ✅ + liveness/readiness probes |
| Ecosystem | Small | Helm, Operators, Service Mesh… |
| Cloud managed | ❌ (*) | EKS, GKE, AKS |


:::{.callout-note .fragment}
(*) AWS ECS / Azure Container Instances/Apps have a similar Swarm-like approach but they are independent services, not Kubernetes distributions.
:::
:::

::::


# Kubernetes Architecture {background-color="white" background-opacity="0.7"}
![Components of Kubernetes](https://kubernetes.io/images/docs/components-of-kubernetes.svg)

## kubeadm: The Bootstrap Tool
<https://kubernetes.io/docs/reference/setup-tools/kubeadm/>

::::{.columns}

::: {.fragment .column width="50%"}

> *"kubeadm performs the actions necessary to get a minimum viable cluster up and running"*
> [kubernetes.io](https://kubernetes.io/docs/reference/setup-tools/kubeadm/)

- Runs **pre-flight checks** (swap off, ≥ 2 CPUs, open ports) [Details](https://kubernetes.io/docs/reference/setup-tools/kubeadm/implementation-details/)
- Generates TLS certificates for all components
- Writes static Pod manifests → `kube-apiserver`, `etcd`, `scheduler`
- Bootstraps a join token for worker nodes

::: {.callout-note .fragment title="KaaS"}
Managed services (EKS, GKE, AKS) run kubeadm under the hood
:::

:::

::: {.fragment .column width="50%"}
Logical steps to set up a cluster with kubeadm:

1. On the control-plane node ```kubeadm init --pod-network-cidr=10.244.0.0/16 --control-plane-endpoint=<IP>```
2. For example, to install [Flannel](https://github.com/flannel-io/flannel): ```kubectl apply -f https://github.com/flannel-io/flannel/releases/latest/download/kube-flannel.yml```
3. For each worker node, kubeadm will output a `kubeadm join` to be run on the worker to join the cluster. It looks like this: ```kubeadm join <IP>:6443 --token <token>```

:::{.callout-note .fragment title="CNI"}
A CNI (Container Network Interface) is required for pod networking, kubeadm does not install one by default, since it depends on the use case and platform.
:::
:::
::::


# Lab Setup {background-color="#2C3E50"}


## Cluster Topology

::::{.columns}

::: {.fragment .column width="50%"}

**Lab directory: `lab/k8s-multipass/`**

```
lab/k8s-multipass/
├── terraform/
│   ├── main.tf          # Multipass VMs + inventory
│   ├── variables.tf     # Counts, CPU, RAM, disk
│   ├── outputs.tf       # IPs + next-steps hint
│   ├── cloud-init.tpl   # SSH key injection
│   ├── inventory.tpl    # Ansible inventory template
│   └── id_ed25519(.pub) # SSH key pair
└── ansible/
    ├── 00-prerequisites.yml
    ├── 01-control-plane.yml
    └── 02-workers.yml
```

:::

::: {.fragment .column width="50%"}

**Default sizing (kubeadm minimum)**

| Node | Count | CPUs | RAM | Disk |
|------|-------|------|-----|------|
| control-plane | 1 | 2 | 2 GB | 10 GB |
| worker | 2 | 2 | 2 GB | 10 GB |

::: {.callout-warning}
`kubeadm` **refuses to initialise** if a node has fewer than 2 CPUs.
The default Multipass sizing (1 CPU / 1 GB) must be overridden in `variables.tf`.
:::

:::

::::


## Provision VMs with OpenTofu

::::{.columns}

::: {.fragment .column width="50%"}

**Step 0 — Generate SSH key pair**

```bash
cd lab/k8s-multipass/terraform
ssh-keygen -t ed25519 -C "k8s-lab" -N "" -f id_ed25519
```

**Step 1 — Init and apply**

```bash
tofu init
tofu plan
tofu apply -auto-approve
```

Expected:
```
multipass_instance.control_plane[0]: Creating...
multipass_instance.worker[0]: Creating...
multipass_instance.worker[1]: Creating...

Apply complete! Resources: 5 added.
```

:::

::: {.fragment .column width="50%"}

**Verify VMs are running**

```bash
multipass list
# Name              State   IPv4
# control-plane-1   Running 192.168.64.10
# worker-1          Running 192.168.64.11
# worker-2          Running 192.168.64.12
```

**Ansible inventory (auto-generated)**

```ini
[control_plane]
control-plane-1 ansible_host=192.168.64.10 ansible_user=ubuntu

[workers]
worker-1 ansible_host=192.168.64.11 ansible_user=ubuntu
worker-2 ansible_host=192.168.64.12 ansible_user=ubuntu

[k8s:children]
control_plane
workers
```

:::

::::


# Node Prerequisites {background-color="#7D3C98"}


## Why Every Node Needs Preparation

::::{.columns}

::: {.fragment .column width="50%"}

**kubeadm pre-flight checks — will fail without:**

| Requirement | Why |
|---|---|
| Swap disabled | kubelet cannot manage memory with swap on |
| ≥ 2 CPUs | control-plane components are CPU-hungry |
| `overlay` module | containerd needs it for layered filesystems |
| `br_netfilter` module | iptables must see bridged traffic |
| IP forwarding on | Pods must route packets between nodes |
| `SystemdCgroup = true` | kubelet + containerd must share the same cgroup driver |

:::

::: {.fragment .column width="50%"}

**Playbook pipeline**

```{mermaid}
flowchart TD
    A["00-prerequisites.yml — all nodes"]:::play
    B["01-control-plane.yml — control_plane"]:::play
    C["02-workers.yml - workers"]:::play
    A --> B --> C

```

::: {.callout-note}
Run the playbooks **in order**  - workers cannot join before the control plane is initialised.
:::

:::

::::


## Ansible: 00-prerequisites.yml

::::{.columns}

::: {.fragment .column width="50%"}

**Targets: `hosts: k8s`** (all nodes)

```yaml
- name: Disable swap immediately
  command: swapoff -a

- name: Load kernel modules (overlay, br_netfilter)
  community.general.modprobe:
    name: "{{ item }}"
  loop: [overlay, br_netfilter]

- name: Set sysctl for Kubernetes networking
  ansible.posix.sysctl:
    name: "{{ item.key }}"
    value: "{{ item.value }}"
  loop:
    - { key: net.bridge.bridge-nf-call-iptables, value: "1" }
    - { key: net.ipv4.ip_forward,                value: "1" }
```

:::

::: {.fragment .column width="50%"}

```yaml
- name: Install containerd and dependencies
  apt:
    name:
      - containerd
      - conntrack   # ← required by kubeadm preflight

- name: Patch SystemdCgroup = true
  shell: |
    containerd config default \
      | sed 's/SystemdCgroup = false/SystemdCgroup = true/' \
      > /etc/containerd/config.toml

- name: Install kubelet, kubeadm, kubectl (pkgs.k8s.io v1.31)
  apt:
    name: [kubelet, kubeadm, kubectl]

- name: Hold packages to prevent accidental upgrades
  dpkg_selections:
    name: "{{ item }}"
    selection: hold
  loop: [kubelet, kubeadm, kubectl]
```

::: {.callout-warning}
`conntrack` is **not** installed by default on Ubuntu 24.04 minimal images. Without it, `kubeadm init` fails at preflight with `[ERROR FileExisting-conntrack]`.
:::

:::

::::


## ✅ Test: Prerequisites

::::{.columns}

::: {.column width="50%"}

**Run on any node via Ansible ad-hoc**

```bash
# Check all nodes pass
ansible k8s -i terraform/hosts.ini \
  -m command -a "swapon --show"
# Expected: no output (swap is off)

ansible k8s -i terraform/hosts.ini \
  -m command -a "lsmod | grep -E 'overlay|br_netfilter'"
# Expected: both modules present on every node

ansible k8s -i terraform/hosts.ini \
  -m command -a "sysctl net.ipv4.ip_forward"
# Expected: net.ipv4.ip_forward = 1
```

:::

::: {.fragment .column width="50%"}

**Check containerd is running**

```bash
ansible k8s -i terraform/hosts.ini \
  -m command -a "systemctl is-active containerd"
# Expected: active  (on all 3 nodes)

ansible k8s -i terraform/hosts.ini \
  -m command -a \
  "grep SystemdCgroup /etc/containerd/config.toml"
# Expected: SystemdCgroup = true
```

**Check kubelet is installed and held**

```bash
ansible k8s -i terraform/hosts.ini \
  -m command -a "kubelet --version"
# Expected: Kubernetes v1.31.x

ansible k8s -i terraform/hosts.ini \
  -m command -a "apt-mark showhold"
# Expected: kubelet  kubeadm  kubectl
```

::: {.callout-tip}
A `hold` on K8s packages means `apt upgrade` will never silently break your cluster.
:::

:::

::::


# Control-Plane Bootstrap {background-color="#1B4F72"}


## kubeadm init — What Happens

::::{.columns}

::: {.fragment .column width="50%"}

```{mermaid}
flowchart TD
    PF["🔍 Pre-flight checks\n(swap, CPUs, ports, modules)"]:::step
    CERT["🔐 Generate TLS certificates\n(CA, apiserver, etcd, kubelet)"]:::step
    STATIC["📄 Write static Pod manifests\n(apiserver, etcd, scheduler, controller-mgr)"]:::step
    BOOT["🪙 Bootstrap token\nfor worker join"]:::step
    CNI["🌐 Wait for CNI\n(apply Flannel)"]:::step
    PF --> CERT --> STATIC --> BOOT --> CNI

    click PF "https://kubernetes.io/docs/reference/setup-tools/kubeadm/kubeadm-init/#before-you-begin" "Checks swap, kernel modules, available ports and container runtime"
    click CERT "https://kubernetes.io/docs/setup/best-practices/certificates/" "Self-signed CA + leaf certs for every component"
    click CNI "https://kubernetes.io/docs/concepts/extend-kubernetes/compute-storage-net/network-plugins/" "Without CNI all Pods stay Pending — we use Flannel"

    classDef step fill:#1B4F72,color:#fff,stroke:#154360
```

:::

::: {.fragment .column width="50%"}

**What `01-control-plane.yml` does after init**

```yaml
- name: Copy admin.conf → ~/.kube/config
  copy:
    src: /etc/kubernetes/admin.conf
    dest: /home/ubuntu/.kube/config
    owner: ubuntu

- name: Install Flannel CNI
  command: kubectl apply -f {{ flannel_manifest }}
  environment:
    KUBECONFIG: /home/ubuntu/.kube/config
```

No manual steps — Ansible handles the full post-init sequence.

::: {.callout-important}
Until Flannel is installed, even system pods in `kube-system` stay `Pending`. This is expected — `kubeadm init` exits successfully before CNI is ready.
:::

:::

::::


## Ansible: 01-control-plane.yml

::::{.columns}

::: {.fragment .column width="50%"}

**Targets: `hosts: control_plane`**

```yaml
- name: Run kubeadm init
  command: >
    kubeadm init
    --pod-network-cidr=10.244.0.0/16
    --control-plane-endpoint={{ ansible_host }}
  when: not kubeadm_init_done.stat.exists

- name: Copy admin.conf → ~/.kube/config
  copy:
    src: /etc/kubernetes/admin.conf
    dest: /home/ubuntu/.kube/config
    owner: ubuntu
```

:::

::: {.fragment .column width="50%"}

```yaml
- name: Install Flannel CNI
  command: kubectl apply -f {{ flannel_manifest }}
  environment:
    KUBECONFIG: /home/ubuntu/.kube/config

- name: Capture join command
  command: kubeadm token create --print-join-command
  register: join_command_raw

- name: Save join command on the controller
  copy:
    content: "{{ join_command_raw.stdout }}"
    dest: /tmp/k8s_join_command.sh
    mode: '0600'
  delegate_to: localhost
  become: false        # ← don't inherit become: true from the play
```

::: {.callout-tip}
`delegate_to: localhost` runs the task on the Ansible controller. `become: false` is required — the play has `become: true` for the remote node, but the controller doesn't need (and may not have) passwordless sudo.
:::

:::

::::


## ✅ Test: Control Plane

::::{.columns}

::: {.column width="50%"}

**Login into the control-plane node**

```bash
multipass exec control-plane-1 -- bash
```

**Inside the VM:**

```bash
kubectl get nodes
# NAME              STATUS   ROLES           AGE
# control-plane-1   Ready    control-plane   3m

kubectl get pods -n kube-system
# NAME                                       READY   STATUS
# coredns-...                                1/1     Running
# etcd-control-plane-1                       1/1     Running
# kube-apiserver-control-plane-1             1/1     Running
# kube-controller-manager-control-plane-1    1/1     Running
# kube-flannel-ds-...                        1/1     Running
# kube-proxy-...                             1/1     Running
# kube-scheduler-control-plane-1             1/1     Running
```

:::

::: {.fragment .column width="50%"}

**What to look for**

| Component | Expected status |
|---|---|
| `kube-apiserver` | `Running` |
| `etcd` | `Running` |
| `kube-flannel-ds` | `Running` |
| `coredns` | `Running` ← needs Flannel |
| Node status | `Ready` |

```bash
# Confirm Flannel assigned a pod CIDR
kubectl get nodes -o jsonpath=\
  '{.items[*].spec.podCIDR}'
# Expected: 10.244.0.0/24
```

::: {.callout-warning}
If `coredns` is `Pending`, Flannel is still initialising. Wait 30 s and retry — it is **not** an error in the playbook.
:::

:::

::::


# Worker Nodes {background-color="#1A5276"}


## Ansible: 02-workers.yml — Join & Verify

::::{.columns}

::: {.fragment .column width="50%"}

**Targets: `hosts: workers`**

```yaml
- name: Check if already joined
  stat:
    path: /etc/kubernetes/kubelet.conf
  register: already_joined

- name: Read join command from controller
  set_fact:
    join_command: "{{ lookup('file',
      '/tmp/k8s_join_command.sh') }}"
  when: not already_joined.stat.exists

- name: Join the cluster
  command: "{{ join_command }}"
  when: not already_joined.stat.exists
```

:::

::: {.fragment .column width="50%"}

**Verify from control-plane**

```bash
kubectl get nodes
# NAME              STATUS   ROLES           AGE
# control-plane-1   Ready    control-plane   5m
# worker-1          Ready    <none>          2m
# worker-2          Ready    <none>          2m

kubectl get nodes -o wide
# also shows IPs and container runtime
```

::: {.callout-note}
The playbook **polls every 15 s** (up to 5 min) until all nodes reach `Ready`. Flannel may take a moment to assign Pod CIDRs to newly joined nodes.
:::

:::

::::


## ✅ Test: Full Cluster


**All nodes Ready**

```bash
# From the control-plane VM
kubectl get nodes -o wide
# NAME              STATUS   ROLES           AGE   VERSION   INTERNAL-IP
# control-plane-1   Ready    control-plane   8m    v1.31.x   192.168.64.10
# worker-1          Ready    <none>          4m    v1.31.x   192.168.64.11
# worker-2          Ready    <none>          4m    v1.31.x   192.168.64.12
```

**Smoke test — run a pod and curl it**

```bash
# Start a test pod
kubectl run test --image=nginx:alpine --port=80

# Wait for it to be Running
kubectl get pod test -w
# test   0/1   ContainerCreating   0   5s
# test   1/1   Running             0   12s

# Get its Pod IP and curl from another pod
POD_IP=$(kubectl get pod test \
  -o jsonpath='{.status.podIP}')
kubectl run curl --image=curlimages/curl \
  --restart=Never --rm -it \
  -- curl -s http://${POD_IP} | head -5
# Expected: <html> nginx welcome page
```

```bash
kubectl delete pod test
```


# kubectl {background-color="white" background-image="https://rnemet.dev/images/k8s_intro_local_and_kubectl.png" background-size="cover" background-opacity="0.7"}
<https://kubernetes.io/docs/reference/kubectl/>

## Core Kubernetes Objects
<https://kubernetes.io/docs/concepts/overview/working-with-objects/>

::::{.columns}
::: {.fragment .column width="50%"}
Kubernetes objects are persistent entities in the Kubernetes system. 

Kubernetes uses these entities to represent the state of your cluster. 
Specifically, they can describe:

- What containerized applications are running (and on which nodes)
- The resources available to those applications
- The policies around how those applications behave, such as restart policies, upgrades, and fault-tolerance
:::

::: {.fragment .column width="50%"}
- A Kubernetes object is a "record of intent"
- Once you create the object, the Kubernetes system will constantly work to ensure that the object exists. 
- By creating an object, you're effectively telling the Kubernetes system what you want your cluster's workload to look like; this is your cluster's desired state.
:::
::::  

## Kubectl

Kubectl is the command-line tool for interacting with Kubernetes clusters. It allows you to run commands against Kubernetes clusters, deploy applications, inspect and manage cluster resources, and view logs.

::::{.columns}

::: {.fragment .column width="50%"}

**Inspect**

```bash
kubectl get nodes                     # cluster nodes
kubectl get pods -A                   # all pods, all namespaces
kubectl get pods -o wide              # with node + IP
kubectl describe pod <name>           # full event log + conditions
kubectl logs <pod>                    # stdout / stderr
kubectl exec -it <pod> -- bash        # shell into pod
```

**Apply / delete**

```bash
kubectl apply -f manifest.yaml        # declarative apply
kubectl delete -f manifest.yaml       # remove by manifest
kubectl delete pod <name>             # remove by name
```

:::

::: {.fragment .column width="50%"}

**Deployments**

```bash
# Create imperatively
kubectl create deployment nginx \
  --image=nginx --replicas=3

# Expose as NodePort
kubectl expose deployment nginx \
  --type=NodePort --port=80

# Scale
kubectl scale deployment nginx --replicas=6

# Rolling update
kubectl set image deployment/nginx nginx=nginx:alpine

# Check rollout progress
kubectl rollout status deployment/nginx

# Rollback
kubectl rollout undo deployment/nginx
```

:::

::::


# Hands-on {background-color="#17202A"}


## Deploy & Expose a Workload

::::{.columns}

::: {.fragment .column width="50%"}

**Deploy nginx across workers**

```bash
# Run kubectl from inside the control-plane VM
multipass exec control-plane-1 -- \
  kubectl create deployment nginx \
    --image=nginx --replicas=3

multipass exec control-plane-1 -- \
  kubectl expose deployment nginx \
    --type=NodePort --port=80

# Get the assigned NodePort
multipass exec control-plane-1 -- \
  kubectl get svc nginx
# NAME    TYPE       CLUSTER-IP   PORT(S)
# nginx   NodePort   10.96.x.x    80:3xxxx/TCP
```

:::

::: {.fragment .column width="50%"}

**Access from the host machine**

```bash
# Get a worker IP from multipass directly
multipass list
# worker-1  Running  192.168.64.11  ...

# Substitute the NodePort shown above
curl http://192.168.64.11:<nodePort>
# <h1>Welcome to nginx!</h1>

# Pods spread across nodes
multipass exec control-plane-1 -- \
  kubectl get pods -o wide
```

::: {.callout-note}
In this manual lab the kubeconfig lives **inside** the control-plane VM (`~/.kube/config`). `kubectl` on your host has no cluster credentials yet — that's what Lesson 10 automates with the infra pipeline.
:::

:::

::::


## Scaling & Rolling Updates

::::{.columns}

::: {.fragment .column width="50%"}

**Scale up**

```bash
multipass exec control-plane-1 -- \
  kubectl scale deployment nginx --replicas=6

multipass exec control-plane-1 -- \
  kubectl get pods -o wide
# Pods spread across both workers
```

**Rolling update — zero downtime**

```bash
multipass exec control-plane-1 -- \
  kubectl set image deployment/nginx \
    nginx=nginx:alpine

multipass exec control-plane-1 -- \
  kubectl rollout status deployment/nginx
# deployment "nginx" successfully rolled out
```

**Rollback**

```bash
multipass exec control-plane-1 -- \
  kubectl rollout undo deployment/nginx
```

:::

::: {.fragment .column width="50%"}

**How rolling updates work**

```{mermaid}
flowchart LR
    OLD["nginx:latest\n6 pods"]:::old
    SURGE["nginx:latest × 5\nnginx:alpine × 1"]:::mid
    MID["nginx:latest × 3\nnginx:alpine × 3"]:::mid
    DONE["nginx:alpine\n6 pods"]:::new

    OLD -->|"set image"| SURGE --> MID --> DONE

    classDef old  fill:#E74C3C,color:#fff,stroke:#a93226
    classDef mid  fill:#F39C12,color:#fff,stroke:#b7770d
    classDef new  fill:#27AE60,color:#fff,stroke:#1a7a42
```

::: {.callout-tip}
By default: `maxSurge: 1`, `maxUnavailable: 0`.  
The old version **keeps serving traffic** until the new pod passes its readiness probe.
:::

:::

::::


## Full Cluster Lifecycle

```{mermaid}
flowchart LR
    OT["🔧 <b>OpenTofu</b><br/><br/>ssh-keygen<br/>tofu apply<br/>&nbsp;"]:::tf
    -->|"3 VMs<br/>provisioned"| MP["☁️ <b>Multipass</b><br/><br/>control-plane-1<br/>worker-1 · worker-2<br/>2 CPU · 2 GB · Ubuntu 24.04"]:::vm
    -->|"SSH ready<br/>inventory"| ANS["📦 <b>Ansible</b><br/><br/>00-prerequisites<br/>01-control-plane<br/>02-workers"]:::ans
    -->|"all nodes<br/>Ready"| K8S["☸️ <b>Kubernetes</b><br/><br/>3 × Ready nodes<br/>Deployment<br/>NodePort Service"]:::k8s

    classDef tf   fill:#7B68EE,color:#fff,stroke:#4b3fa0,font-size:18px
    classDef vm   fill:#27AE60,color:#fff,stroke:#1a7a42,font-size:18px
    classDef ans  fill:#E74C3C,color:#fff,stroke:#a93226,font-size:18px
    classDef k8s  fill:#326CE5,color:#fff,stroke:#1a4a9f,font-size:18px
```


## Cleanup

::::{.columns}

::: {.fragment .column width="50%"}

**Remove workloads**

```bash
multipass exec control-plane-1 -- \
  kubectl delete deployment nginx

multipass exec control-plane-1 -- \
  kubectl delete service nginx
```

**Destroy all VMs**

```bash
cd lab/k8s-multipass/terraform
tofu destroy -auto-approve
```

Expected:
```
multipass_instance.worker[0]: Destroying...
multipass_instance.worker[1]: Destroying...
multipass_instance.control_plane[0]: Destroying...

Destroy complete! Resources: 5 destroyed.
```

:::

::: {.fragment .column width="50%"}

**Verify nothing remains**

```bash
multipass list
# No instances found.
```

::: {.callout-important}
Always `tofu destroy` when you're done.  
Each running VM consumes host CPU + RAM.  
Multipass VMs are **not** automatically cleaned up on reboot.
:::

:::

::::


## Summary

| Step | Tool | Command |
|------|------|---------|
| Generate SSH key | ssh-keygen | `ssh-keygen -t ed25519 -f id_ed25519` |
| Provision VMs | OpenTofu | `tofu apply` |
| Generate inventory | Terraform template | *(automatic)* |
| Install prerequisites | Ansible | `ansible-playbook 00-prerequisites.yml` |
| Init control plane | Ansible + kubeadm | `ansible-playbook 01-control-plane.yml` |
| Join workers | Ansible + kubeadm | `ansible-playbook 02-workers.yml` |
| Verify cluster | kubectl | `kubectl get nodes` |
| Deploy workload | kubectl | `kubectl create deployment …` |
| Expose service | kubectl | `kubectl expose deployment … --type=NodePort` |
| Scale | kubectl | `kubectl scale deployment … --replicas=N` |
| Destroy | OpenTofu | `tofu destroy` |
